# Playground: define, train and explore an experiment

A scratch notebook for the experiment pipeline (`fct.exp`), end to end: define an experiment from a
preset, train its settings here in the kernel with the log streaming below the cell, load one
trained setting and try analysis functions on it.

Edit the parameters cell and run top to bottom. Training resumes where it stopped and skips
finished settings, so re-running is cheap. `ROOT` holds the experiment; a different experiment in
the same `ROOT` is refused, so change `ROOT` (or delete it) to start another.

In [ ]:
ROOT = None  # experiment directory; None: $FCT_PLAYGROUND_ROOT, else <repo>/tmp/playground (git-ignored)
PRESET = "tiny"  # base preset, see fct.exp.preset_names()
OVERRIDES = {"n_epochs": 20000, "N": 32, "P": 256, "log_int": 2000, "save_int": 5000, "monitor_int": 2000}  # base-field overrides
AXES = {"net.L": [1, 2]}  # axis name -> values (one setting per combination); None keeps the preset's axes
DEVICE = "cpu"  # "cpu", "cuda" or "auto"
SETTING = 0  # the setting to load below: an index, or a dict of axis values such as {"net.L": 2}
MEMBER = None  # which network of an ensemble (net.B > 1) to load; None for a single network

## Define the experiment

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

import fct
from fct.analyse import example_outputs, r_squared
from fct.config import resolve_dtype
from fct.data import get_dataset
from fct.exp import (
    ExpConfig,
    exp_path,
    expand,
    get_setting,
    load_exp,
    preset,
    run_setting,
    same_experiment,
    save_exp,
    setting_config,
    setting_dir,
    with_axes,
)
from fct.spatial_exp import select_setting, settings_table
from fct.train import load_trained, select_member

REPO = Path(fct.__file__).resolve().parents[1]
ROOT = Path(ROOT or os.environ.get("FCT_PLAYGROUND_ROOT") or REPO / "tmp" / "playground")

In [ ]:
base = preset(PRESET)
exp = ExpConfig.model_validate({**base.model_dump(), **OVERRIDES, "device": DEVICE})
if AXES is not None:
    exp = with_axes(exp, AXES)

# Never overwrite an experiment already in ROOT: its trained settings belong to it.
if exp_path(ROOT).exists() and not same_experiment(load_exp(ROOT), exp):
    raise ValueError(
        f"{ROOT} already holds a different experiment; change ROOT or delete that directory"
    )
save_exp(exp, ROOT)
display(Markdown(settings_table(ROOT)))

## Train

Each setting trains in this kernel; the log lines appear as it runs.

In [ ]:
# run_setting resumes a partial run and skips a finished one, so re-running this cell is cheap.
n_settings = len(expand(exp))
for index in range(n_settings):
    run_setting(exp, index, ROOT, log=print)

In [ ]:
fig, ax = plt.subplots()
for index in range(n_settings):
    setting = get_setting(exp, index)
    cfg = setting_config(exp, setting)
    _, saved = load_trained(
        setting_dir(ROOT, setting), cfg, torch.device("cpu"), resolve_dtype(cfg.dtype)
    )
    loss = np.asarray(saved["loss"], dtype=float)  # an ensemble has a member axis: average it
    ax.plot(saved["epoch"], loss.mean(axis=tuple(range(1, loss.ndim))), label=str(setting.axes))
ax.set(xlabel="epoch", ylabel="loss")
ax.legend()
plt.show()

## Load one trained setting

`SETTING` picks it, an index or axis values.

In [ ]:
axes = get_setting(exp, SETTING).axes if isinstance(SETTING, int) else SETTING
chosen = select_setting(ROOT, axes, DEVICE)
if chosen.status != "done":
    raise RuntimeError(f"setting {dict(axes)} is {chosen.status}; run the train cell first")
cfg = chosen.cfg
net, history = select_member(
    *load_trained(chosen.directory, cfg, torch.device(DEVICE), resolve_dtype(cfg.dtype)), MEMBER
)
data = get_dataset(chosen.directory, cfg)
print(dict(axes), "| x", tuple(data.x.shape), "| y", tuple(data.y.shape))
print("history keys:", sorted(history))

## Analysis playground

Library functions to call on `cfg`, `net`, `data` and `history` (see `help(...)`):

- `fct.analyse`: `example_outputs`, `pick_examples`, `r_squared`, `r_squared_per_output`, `prediction_report`, `grid_predictions`, `layer_pca`.
- `fct.tuning`: `tuning_curves`, `select_neurons`.
- `fct.spatial`: `build_spaces`, `data_kernel`, `latent_kernel`, `spatial_kernel_pairs`.
- `fct.phase_view`: `layer_priors`, `member_view`, `ensemble_views`, `average_views` (for ensembles).
- `fct.exp_view`: `tuning_view`, `layer_rates`, `neuron_information`.

In [ ]:
everything = example_outputs(net, data, torch.arange(len(data.x)))
print("R^2 on the training data:", r_squared(everything.y_true, everything.y_pred))

In [ ]:
fig, ax = plt.subplots()
ax.scatter(everything.y_true.flatten(), everything.y_pred.flatten(), s=4)
ax.set(xlabel="target", ylabel="network output")
plt.show()